Evaluate the presence of factual claim-making in SemEval articles and the relationship between propaganda and claim-making.

In [19]:
import pandas as pd
import spacy
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

In [20]:
#Import functions for claim extraction
project_root = Path(os.getcwd()).parent
utils_path = str(project_root / "create-model")
if utils_path not in sys.path:
    sys.path.append(utils_path)
from claim_utils import lightweight_claimify

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/frankiepike/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/frankiepike/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
04/06/2026 13:01:34 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/06/2026 13:01:34 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/config.json "HTTP/1.1 200 OK"
04/06/2026 13:01:34 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/06/2026 13:01:34 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/config.json "HTTP/1.1 200 OK"
04/06/2026 13:01:34 - INFO - 	 HTTP Request: H

Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

FCorefModel LOAD REPORT from: biu-nlp/f-coref
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
04/06/2026 13:01:35 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref "HTTP/1.1 200 OK"
04/06/2026 13:01:35 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/commits/main "HTTP/1.1 200 OK"
04/06/2026 13:01:35 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/discussions?p=0 "HTTP/1.1 200 OK"
04/06/2026 13:01:35 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
04/06/2026 13:01:35 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Fou

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

04/06/2026 13:01:36 - INFO - 	 HTTP Request: HEAD https://huggingface.co/valhalla/distilbart-mnli-12-3/resolve/refs%2Fpr%2F1/model.safetensors "HTTP/1.1 302 Found"
04/06/2026 13:01:36 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
04/06/2026 13:01:36 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/valhalla/distilbart-mnli-12-3/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


In [21]:
#Load lightweight spaCy model
nlp = spacy.load("en_core_web_sm")

In [22]:
#Load cleaned SemEval datasets
BASE_DIR = Path("../").resolve()
DATA_PATH = BASE_DIR / "data" / "interim" / "semeval_task2_tc_merged.csv"
df_merged = pd.read_csv(DATA_PATH)

In [23]:
#Group gold spans and unique articles
gold_spans_dict = df_merged.groupby('article_id').apply(
    lambda x: list(zip(x['start_char'].astype(int), x['end_char'].astype(int)))
).to_dict()

unique_articles = df_merged[['article_id', 'text_content']].drop_duplicates('article_id')

In [24]:
#Analysis logic
def has_overlap(sent_start, sent_end, spans):
    for span_start, span_end in spans:
        if span_start < sent_end and span_end > sent_start:
            return True
    return False

In [25]:
analysis_results = []
counts = {"Propaganda Only": 0, "Fact Only": 0, "Both": 0, "Neither": 0}

In [ ]:
for _, row in tqdm(unique_articles.iterrows(), total=len(unique_articles)):
    aid = row['article_id']
    text = str(row['text_content'])
    spans = gold_spans_dict.get(aid, [])

    doc = nlp(text)
    art_prop_sents = 0
    art_fact_sents = 0
    total_sents = 0

    for sent in doc.sents:
        total_sents += 1
        s_start, s_end = sent.start_char, sent.end_char

        #Check Gold Propaganda (from dataset)
        is_prop = has_overlap(s_start, s_end, spans)

        #Check Factual Claim (using your spacy_decompose from 4.0)
        claims = lightweight_claimify(sent.text)
        is_fact = len(claims) > 0

        if is_prop: art_prop_sents += 1
        if is_fact: art_fact_sents += 1

        #Categorize for the global overlap metric
        if is_prop and is_fact:
            counts["Both"] += 1
        elif is_prop:
            counts["Propaganda Only"] += 1
        elif is_fact:
            counts["Fact Only"] += 1
        else:
            counts["Neither"] += 1

    #Store article-level density for correlation analysis
    if total_sents > 0:
        analysis_results.append({
            "article_id": aid,
            "prop_density": (art_prop_sents / total_sents) * 100,
            "fact_density": (art_fact_sents / total_sents) * 100
        })

df_stats = pd.DataFrame(analysis_results)
display(df_stats)

  0%|          | 0/357 [00:00<?, ?it/s]04/06/2026 13:02:16 - INFO - 	 Tokenize 1 inputs...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

04/06/2026 13:02:16 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]

04/06/2026 13:02:19 - INFO - 	 Tokenize 1 inputs...


...

Inference:   0%|          | 0/1 [00:00<?, ?it/s]

04/06/2026 13:04:01 - INFO - 	 Tokenize 1 inputs...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

04/06/2026 13:04:01 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#1. Percentage Breakdown (The Intersections)
plt.figure(figsize=(10, 7))
plt.pie(counts.values(), labels=counts.keys(), autopct='%1.1f%%', startangle=140, colors=sns.color_palette("viridis"))
plt.title("Distribution of Sentences: Propaganda vs. Factual Claims")
plt.show()

In [ ]:
#2. Correlation Analysis (Propaganda vs. Facts)
plt.figure(figsize=(10, 6))
correlation = df_stats['prop_density'].corr(df_stats['fact_density'])
sns.regplot(data=df_stats, x='prop_density', y='fact_density',
            scatter_kws={'alpha':0.4}, line_kws={'color':'red'})

plt.title(f"Correlation: {correlation:.3f}\nPropaganda Density vs. Fact Density per Article")
plt.xlabel("% of Sentences containing Propaganda")
plt.ylabel("% of Sentences containing Factual Claims")
plt.grid(True, alpha=0.3)
plt.show()